# Vi-PPE-mini Phase 11 - Colab A100 Runner

Notebook này chạy Phase 11 trên Google Colab A100 cho repo `KhanhTruong05/Vi-PPE-mini`.

Luồng khuyến nghị:
1. Kiểm tra GPU.
2. Clone repo và cài dependencies.
3. Kiểm tra dataset processed.
4. Chạy smoke real-model với `--limit 5`.
5. Chạy full Phase 11.
6. Lưu `results/` về Google Drive.

In [ ]:
!nvidia-smi

## 1. Clone Repo

Nếu repo private, đổi URL clone sang dạng có token GitHub.

In [ ]:
%cd /content
!rm -rf Vi-PPE-mini
!git clone https://github.com/KhanhTruong05/Vi-PPE-mini.git
%cd /content/Vi-PPE-mini
!git rev-parse --short HEAD

## 2. Install Dependencies

In [ ]:
!python -m pip install -U pip
!pip install -r requirements.txt
!pip install -U bitsandbytes accelerate transformers datasets peft trl

## 3. Verify Data And Code

Cần có `pairs_dev.jsonl`, `pairs_test.jsonl`, `bias_subset.jsonl`, `dataset_manifest.json` trong `data/processed/`.

In [ ]:
!ls -lah data/processed
!python scripts/00_smoke_check.py
!pytest -q

Nếu dataset không nằm trong Git, mount Drive rồi copy vào `data/processed/`.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p data/processed
# !cp -r /content/drive/MyDrive/Vi-PPE-mini/data/processed/* data/processed/

## 4. Smoke Real Model Run

Chạy mỗi run 5 pair trước. Nếu cell này pass, mới chạy full.

In [ ]:
!python scripts/11_colab_phase11_runner.py --limit 5

## 5. Full Phase 11 Run

Sau smoke, chạy full. Script dùng `--resume`, nên nếu Colab disconnect có thể chạy lại cell này.

In [ ]:
!python scripts/11_colab_phase11_runner.py --skip-preflight

## 6. Metrics Only

Dùng cell này nếu inference đã xong nhưng cần tính lại metrics.

In [ ]:
!python scripts/11_colab_phase11_runner.py --skip-preflight --skip-inference

## 7. Save Results To Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/Vi-PPE-mini/results
!cp -r results/judgments /content/drive/MyDrive/Vi-PPE-mini/results/
!cp -r results/metrics /content/drive/MyDrive/Vi-PPE-mini/results/
!cp -r results/figures /content/drive/MyDrive/Vi-PPE-mini/results/
!find /content/drive/MyDrive/Vi-PPE-mini/results -maxdepth 2 -type f | sort | tail -100

## 8. Useful Single-Run Commands

Chạy một run cụ thể nếu cần debug.

In [ ]:
# !python scripts/11_colab_phase11_runner.py --only-run qwen25_baseline_dev --limit 5
# !python scripts/11_colab_phase11_runner.py --only-run qwen25_bias_mitigated --skip-preflight